# Accurate sum and dot product

How accurate is a dot product? With the plain loop, it depends on the condition
number $\operatorname{cond}(x^T y) = 2\,|x|^T|y| \,/\, |x^T y|$: each decade of
it costs a correct digit, so in double precision a dot product with condition
number $10^{16}$ has none left.

Ogita, Rump and Oishi ("Accurate sum and dot product", *SIAM J. Sci. Comput.*
26(6), 2005) compute sums and dot products **as if in K-fold working
precision**, with working-precision arithmetic only. Error-free transformations
(`TwoSum`, `TwoProduct`) capture every rounding error, and the errors are summed
again, K − 1 times. Section 6 of the paper tests the algorithms on data from its
own generator, GenDot.

This notebook reruns those experiments with MTL5's implementations, checks every
result against the exact value, and checks the paper's error bounds on every
sample.

**Uses:**
- `mtl5.dot_k` and `mtl5.sum_k`: DotK and SumK.
- `mtl5.mixed.dot` with `accumulator=`:
  - `None`: the plain loop.
  - `'fma64'`: each product fused into the sum.
  - `'compensated'`: Dot2.
  - `'exact'`: the exact sum, rounded once.
- `mtl5.generators.gen_dot` and `gen_sum`: GenDot, the paper's Algorithm 6.1.

**From C++:** [`accurate_sum_dot/accurate_sum_dot.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/accurate_sum_dot/accurate_sum_dot.cpp).

In [ ]:
import math
import time
from fractions import Fraction

import matplotlib.pyplot as plt
import numpy as np

import mtl5

# Chart styling: recessive axes and grid, ink for text, validated categorical hues.
SURFACE = "#fcfcfb"
INK, INK_2 = "#0b0b0b", "#52514e"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#d8d7d2",
        "axes.labelcolor": INK_2,
        "axes.titlecolor": INK,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.color": "#e8e7e2",
        "grid.linewidth": 0.8,
        "text.color": INK,
        "xtick.color": INK_2,
        "ytick.color": INK_2,
        "font.size": 10,
        "figure.dpi": 120,
    }
)

U64, U32 = 2.0**-53, 2.0**-24  # the paper's eps: the unit roundoff
gen = mtl5.generators

## Measuring the error exactly

A relative error of $10^{-17}$ can't be measured against a reference computed
in double. The reference here is exact:

- **float64:** each product $x_i y_i$ is split, without error, into four
  products of half-length significands (Dekker's algorithm).
- **float32:** a product of two floats is already exact in double.

`math.fsum` then rounds the exact sum of those terms once. The residual
$x^T y - \text{res}$ is computed the same way, with $-\text{res}$ as one more
term, so only the difference is rounded. As in the paper, the plots draw a
relative error above 2 at 2. The checks use the errors as computed.

In [ ]:
SPLIT = 2.0**27 + 1  # Veltkamp's splitting constant for float64


def exact_terms(x, y):
    # Floats whose exact sum is x . y. Assumes no overflow in the split, which
    # holds for GenDot's data up to the condition numbers used here.
    x, y = np.asarray(x), np.asarray(y)
    if x.dtype == np.float32:  # a product of two floats is exact in float64
        return x.astype(np.float64) * y.astype(np.float64)
    cx, cy = SPLIT * x, SPLIT * y
    xh, yh = cx - (cx - x), cy - (cy - y)  # the high 26 bits
    xl, yl = x - xh, y - yh  # the rest, exactly
    return np.concatenate([xh * yh, xh * yl, xl * yh, xl * yl])


class Reference:
    # The exact value v of a sum of float terms, as hi + lo: hi = fsum(terms)
    # rounded once, and lo = the rest, rounded once. residual(res) is v - res
    # to within u^2 |v|, far below any error shown here.
    def __init__(self, terms):
        terms = list(map(float, terms))
        self.terms = terms
        self.hi = math.fsum(terms)
        self.lo = math.fsum(terms + [-self.hi])

    def residual(self, res):
        return math.fsum([self.hi, -float(res), self.lo])

    def relerr(self, res):  # not capped: the bounds are checked on this
        return abs(self.residual(res)) / abs(self.hi)

    def correctly_rounded(self, res, dtype):
        # Decided exactly: v - res as a Fraction. Every float is a dyadic
        # rational, so the terms sum exactly as integers over a common power
        # of two. res is v rounded to nearest-even in dtype when v lies within
        # half the gap to res's neighbor on v's side, or exactly halfway with
        # res's last significand bit zero.
        pairs = [t.as_integer_ratio() for t in self.terms + [-float(res)]]
        den = max(q for _, q in pairs)
        d = Fraction(sum(p * (den // q) for p, q in pairs), den)
        if d == 0:
            return True
        T = np.dtype(dtype).type
        neighbor = np.nextafter(T(res), T(np.inf) if d > 0 else T(-np.inf))
        half_gap = abs(Fraction(float(neighbor)) - Fraction(float(res))) / 2
        if abs(d) != half_gap:
            return abs(d) < half_gap
        bits = np.array(res, dtype=dtype).view(np.uint64 if T is np.float64 else np.uint32)
        return int(bits) % 2 == 0


def gamma(n, u):
    return n * u / (1 - n * u)

## The algorithms and their bounds

Each relative-error bound is a function of the length $n$, the condition number
and the unit roundoff $u$:

| algorithm | mtl5 | bound |
| --- | --- | --- |
| plain loop | `mixed.dot(x, y)` | $\tfrac12 \gamma_n \operatorname{cond}$ |
| FMA loop | `mixed.dot(x, y, accumulator='fma64')` | $\tfrac12 \gamma_n \operatorname{cond}$ |
| Dot2 | `dot_k(x, y, 2)`, `mixed.dot(x, y, accumulator='compensated')` | $u + \tfrac12 \gamma_n^2 \operatorname{cond}$ (5.8) |
| DotK | `dot_k(x, y, K)` | $u + 2\gamma_{4n-2}^2 + \tfrac12 \gamma_{4n-2}^K \operatorname{cond}$ (5.11) |
| exact | `mixed.dot(x, y, accumulator='exact')` | $u$ |

with $\gamma_n = nu / (1 - nu)$.

In [ ]:
def algorithms(dtype, k_max):
    fma = "fma64" if dtype == "f64" else "fma32"
    algs = {
        "plain": (lambda x, y: mtl5.mixed.dot(x, y, result="element"), "loop"),
        "fma": (lambda x, y: mtl5.mixed.dot(x, y, accumulator=fma, result="element"), "loop"),
        "Dot2": (lambda x, y: mtl5.dot_k(x, y, 2), 2),
    }
    for k in range(3, k_max + 1):
        algs[f"Dot{k}"] = (lambda x, y, k=k: mtl5.dot_k(x, y, k), k)
    algs["exact"] = (
        lambda x, y: mtl5.mixed.dot(x, y, accumulator="exact", result="element"),
        "exact",
    )
    return algs


def bound(kind, n, cond, u):
    if kind == "loop":
        return gamma(n, u) * cond / 2
    if kind == "exact":
        return u
    if kind == 2:
        return u + gamma(n, u) ** 2 * cond / 2
    g = gamma(4 * n - 2, u)
    return u + 2 * g * g + g**kind * cond / 2


def sweep(n, samples, max_log10, k_max, dtype="f64"):
    # GenDot products with requested condition numbers spread evenly in
    # exponent over [1, 10^max_log10]; seed k for sample k.
    u = U64 if dtype == "f64" else U32
    algs = algorithms(dtype, k_max)
    rows = {name: [] for name in algs}
    conds, dot2_same, rounded, skipped = [], True, True, 0
    np_dtype = np.float64 if dtype == "f64" else np.float32
    for k in range(samples):
        g = gen.gen_dot(n, 10.0 ** (max_log10 * k / (samples - 1)), seed=k, dtype=dtype)
        if not math.isfinite(g.cond):
            skipped += 1  # x . y == 0: no relative error
            continue
        ref = Reference(exact_terms(g.x, g.y))
        conds.append(g.cond)
        for name, (run, _) in algs.items():
            res = run(g.x, g.y)
            rows[name].append(ref.relerr(res))
            if name == "exact":
                rounded &= ref.correctly_rounded(res, np_dtype)
        compensated = mtl5.mixed.dot(g.x, g.y, accumulator="compensated", result="element")
        dot2_same &= mtl5.dot_k(g.x, g.y, 2) == compensated
    conds = np.array(conds)
    errs = {name: np.array(v) for name, v in rows.items()}
    bounds = {
        name: np.array([bound(kind, n, c, u) for c in conds]) for name, (_, kind) in algs.items()
    }
    return conds, errs, bounds, dot2_same, rounded, skipped


t0 = time.perf_counter()
c100, e100, b100, same100, round100, skip100 = sweep(100, 1000, 120, 7)
c2000, e2000, b2000, same2000, round2000, skip2000 = sweep(2000, 200, 120, 7)
c32, e32, b32, same32, round32, skip32 = sweep(100, 1000, 36, 4, dtype="f32")
print(
    f"{len(c100) + len(c2000) + len(c32)} GenDot products in {time.perf_counter() - t0:.1f} s; "
    f"skipped (x . y = 0): {skip100 + skip2000 + skip32}"
)

## Dot2 and DotK against the condition number (Figures 6.1–6.3)

Each point is one GenDot product of length 100. A relative error of exactly 0
can't go on a log axis, so it is drawn at $10^{-19}$. The dashed line is each
algorithm's bound, and the dotted line is $u = 2^{-53}$.

In [ ]:
FLOOR = 1e-19


def scatter(ax, conds, errs, bounds, name, color, label=None, show_bound=True, size=7):
    ax.scatter(
        conds,
        np.clip(errs[name], FLOOR, 2.0),
        s=size,
        color=color,
        alpha=0.6,
        linewidths=0,
        label=label or name,
    )
    if show_bound:
        order = np.argsort(conds)
        ax.plot(conds[order], np.minimum(bounds[name][order], 1e3), color=color, ls="--", lw=1.2)


def setup(ax, title, u, top=120, step=20):
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_ylim(5e-20, 10)
    ax.set_xlim(0.5, 10.0 ** (top + step / 2))
    ax.set_xticks([10.0**e for e in range(0, top + 1, step)])
    ax.axhline(u, color=INK_2, lw=0.8, ls=":")
    ax.set_xlabel("condition number")
    ax.set_ylabel("relative error")
    ax.set_title(title, fontsize=10)
    ax.grid(True, which="major")


fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharey=True)
for ax, (name, color) in zip(axes, [("Dot2", SERIES[0]), ("Dot3", SERIES[1]), ("Dot4", SERIES[2])]):
    scatter(ax, c100, e100, b100, name, color)
    setup(ax, f"{name}, float64, n = 100", U64)
    ax.label_outer()
fig.tight_layout()
plt.show()

Dot2 is as accurate as the rounded exact result up to a condition number of
about $1/u \approx 10^{16}$. Between $1/u$ and $1/u^2$, its error grows in
proportion to the condition number until no digit is left. DotK moves that curve
about 16 decades to the right with each step of K: the algorithm behaves as if
computed in K-fold precision and rounded once.

The bounds hold with room to spare. Like the paper, which calls (5.11)
increasingly pessimistic as K grows, the gap widens with K.

## The plain loop, and the FMA loop

The plain loop loses a digit with every decade of condition number, from the
start. The FMA loop rounds half as often and is no more accurate: what limits it
is the condition number, not the number of roundings.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.2))
for name, color, label in [
    ("plain", INK_2, "plain loop"),
    ("fma", SERIES[3], "FMA loop"),
    ("Dot2", SERIES[0], "Dot2"),
]:
    scatter(ax, c100, e100, b100, name, color, label, show_bound=name != "fma", size=6)
setup(ax, "float64, n = 100; dashed: the loop's bound and (5.8)", U64, top=40, step=10)
ax.legend(fontsize=8, markerscale=2.5, loc="lower right")
plt.show()

## Length 2000, K up to 7 (Figure 6.4)

The paper repeats the test at length 2000 with K up to 7, and observes that
Dot7 still gives the maximum accuracy up to a condition number of $10^{93}$.
Below, for each algorithm: the largest condition number at which its error is
still at most $u$, and the smallest at which it has no correct digit left.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.4))
names = ["Dot2"] + [f"Dot{k}" for k in range(3, 8)]
palette = SERIES + ["#8a5cc4", "#c2477f"]
for name, color in zip(names, palette):
    scatter(ax, c2000, e2000, b2000, name, color, show_bound=False, size=6)
setup(ax, "Dot2 and DotK, float64, n = 2000", U64)
ax.legend(fontsize=8, markerscale=2.5, loc="lower right")
plt.show()


def last_accurate(conds, errs):
    ok = conds[errs <= U64]
    return ok.max() if ok.size else float("nan")


def first_lost(conds, errs):
    bad = conds[errs >= 1]
    return bad.min() if bad.size else float("nan")


print(
    f"{'':6} {'n = 100: err <= u up to':>24} {'err >= 1 from':>14} {'n = 2000: err <= u up to':>25}"
)
for name in names:
    print(
        f"{name:6} {last_accurate(c100, e100[name]):24.1e} {first_lost(c100, e100[name]):14.1e} "
        f"{last_accurate(c2000, e2000[name]):25.1e}"
    )

## In float32

The paper works in double only. In float32, with $u = 2^{-24}$, the same
algorithms scale as the theory says:

- the plain loop fails near $10^{7}$, Dot2 near $10^{14}$, and Dot3 and Dot4
  near $10^{21}$ and $10^{28}$;
- the condition numbers stop near $10^{37}$, the limit of float32's exponent
  range for GenDot.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.2))
for name, color in [
    ("plain", INK_2),
    ("Dot2", SERIES[0]),
    ("Dot3", SERIES[1]),
    ("Dot4", SERIES[2]),
    ("exact", SERIES[3]),
]:
    scatter(ax, c32, e32, b32, name, color, show_bound=name not in ("plain", "exact"), size=6)
setup(ax, "float32, n = 100; dashed: (5.8) and (5.11)", U32, top=35, step=5)
ax.set_ylim(1e-10, 10)
ax.legend(fontsize=8, markerscale=2.5, loc="lower right")
plt.show()

## Summation: correct digits (Table 6.2)

The paper's Table 6.2 lists correct decimal digits, $-\log_{10}$ of the
relative error, set to 0 when negative. Rows are condition numbers $10^7$,
$10^{14}$, $10^{21}$ and $10^{28}$; an exact result counts as 16 digits. The sums
come from `gen_sum`.

For a sum, the reference is free: the summands are floats, so `math.fsum`
gives the exact residual $\sum p_i - \text{res}$ rounded once.

GenDot usually overshoots the requested condition number by about a factor of
10, and each decade costs a digit. So the table shows the median *achieved*
condition number next to each requested one.

In [ ]:
def digits(rel):
    return 16.0 if rel == 0 else min(max(-math.log10(rel), 0.0), 16.0)


SUMS = {"sum": 1, "Sum2": 2, "Sum3": 3}
lengths = range(100, 5001, 300)
table = {}
for c in (1e7, 1e14, 1e21, 1e28):
    achieved, got = [], {name: [] for name in SUMS}
    for n in lengths:
        for s in range(4):
            g = gen.gen_sum(n, c, seed=1000 * n + s)
            exact = math.fsum(g.p)
            achieved.append(g.cond)
            for name, k in SUMS.items():
                res = mtl5.sum_k(g.p, k)
                got[name].append(digits(min(abs(math.fsum(list(g.p) + [-res])) / abs(exact), 2.0)))
    table[c] = (np.median(achieved), {name: np.median(v) for name, v in got.items()})

print(f"{'cond':>6} {'achieved':>9}" + "".join(f"{name:>7}" for name in SUMS))
for c, (ach, med) in table.items():
    print(f"{c:6.0e} {ach:9.1e}" + "".join(f"{med[name]:7.1f}" for name in SUMS))

Plain summation keeps about $16 - \log_{10}\operatorname{cond}$ digits. Sum2
is as accurate as the working precision up to about $10^{16}$, and Sum3 is
fully accurate throughout.
In the cells that depend on the condition number, the paper's values are about
a digit higher. The cause is the overshoot above: the paper's rows look
labelled by the condition number reached, not the one requested.

## What it costs

Below are times relative to the plain loop, for one length, on random
vectors: the cost doesn't depend on the condition number. They depend on the
machine, so they are shown, not checked. The ratios are larger than those the
C++ example reports: its base is MTL5's scalar loop, and the plain loop here runs
about as fast as `mtl5.dot`, MTL5's vectorized path. `dot_k` with K ≥ 3 also
allocates a copy of its 2n error terms.

In [ ]:
def best_us(fn, repeats=7):
    best = float("inf")
    for _ in range(repeats):
        t = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - t)
    return best * 1e6


n_t = 200_000
rng = np.random.default_rng(65)  # the cost doesn't depend on the condition number
x, y = rng.standard_normal(n_t), rng.standard_normal(n_t)
algs = algorithms("f64", 5)
times = {"mtl5.dot": best_us(lambda: mtl5.dot(x, y))}
times.update({name: best_us(lambda run=run: run(x, y)) for name, (run, _) in algs.items()})
base = times["plain"]
fig, ax = plt.subplots(figsize=(7.2, 3.4))
names = list(times)
ax.barh(names, [times[k] / base for k in names], color=SERIES[0])
for i, k in enumerate(names):
    ax.text(times[k] / base, i, f" {times[k] / base:.1f}", va="center", fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("time relative to the plain loop")
ax.set_title(f"dot products, float64, n = {n_t:,}", fontsize=10)
fig.tight_layout()
plt.show()

## Check

These are the claims the figures make, asserted:

- **Bounds:** every one of the 2200 dot products of the three sweeps has a
  finite result (none was skipped for a zero dot product), and no algorithm
  exceeds its bound on any of them. The errors checked are not capped at 2.
- **Dot2:** `dot_k(x, y, 2)` and the compensated accumulator compute the same
  loop, and give the same bits on every sample.
- **Exact:** the exact accumulator's result is correctly rounded, to nearest
  with ties to even, decided in exact rational arithmetic.
- **Each K buys about 16 decades:** in float64 at n = 100, DotK is still
  accurate to $u$ past a condition number of $10^{16(K-1)}$. At n = 2000 the
  bound's $\gamma_n$ is 20 times larger, and the curves start a little
  earlier.
- **Table 6.2:**
  - Sum3 is fully accurate at every condition number.
  - Sum2 keeps at least 15.5 digits up to $10^{14}$. It isn't correctly
    rounded, so its error can be a little above $u$.
  - Plain summation has no digit left from $10^{21}$.

In [ ]:
for label, (errs, bounds) in {
    "n = 100": (e100, b100),
    "n = 2000": (e2000, b2000),
    "float32": (e32, b32),
}.items():
    for name in errs:
        assert np.isfinite(errs[name]).all(), f"{label}: {name} has a non-finite result"
        over = np.count_nonzero(errs[name] > bounds[name])
        assert over == 0, f"{label}: {name} exceeds its bound on {over} samples"
assert same100 and same2000 and same32, "Dot2 and the compensated accumulator differ"
assert skip100 == skip2000 == skip32 == 0 and len(c100) + len(c2000) + len(c32) == 2200
assert round100 and round2000 and round32, "the exact accumulator is not correctly rounded"

for k in range(2, 8):
    assert last_accurate(c100, e100[f"Dot{k}"]) > 10.0 ** (16 * (k - 1)), k

med = {c: table[c][1] for c in table}
assert all(med[c]["Sum3"] == 16.0 for c in med)
assert med[1e7]["Sum2"] >= 15.5 and med[1e14]["Sum2"] >= 15.5
assert med[1e21]["sum"] == med[1e28]["sum"] == 0.0
print("all checks passed")

## From C++

[`accurate_sum_dot/accurate_sum_dot.cpp`](https://github.com/stillwater-sc/mtl5/blob/v5.25.0/examples/accurate_sum_dot/accurate_sum_dot.cpp) runs the same experiments, and writes CSV files that its
`plot_accurate_sum_dot.py` draws. Differences from it:

- **The exact reference:** the C++ subtracts each result inside an exact
  accumulator. Here the products are split exactly, and `math.fsum` sums the
  parts. Both round only the final difference.
- **The paper's comparison algorithms** (Kahan–Babuška summation, and XBLAS's
  double-double sum and dot product) are local to the C++ example, so they are
  not here.
- **Sample counts:** the n = 2000 sweep draws 200 samples instead of 1000 (GenDot's
  cost grows with the square of n), and
  Table 6.2 uses every third length. Both keep the notebook inside the docs
  build's time budget.
- **Timing:** the C++ example times every algorithm at three lengths; here, the
  dot products at one length.